In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [4]:
!pip uninstall -y torchao -q
!pip install -q -U peft transformers datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 91.7 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 33.9 MB/s eta 0:00:00


In [5]:

from pathlib import Path

import pandas as pd
import torch
from transformers import (
    AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer, set_seed
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

KAGGLE_DATA_DIR = Path("/kaggle/input/competitions/smart-mcq-solver-challenge")
DATA_DIR = KAGGLE_DATA_DIR if KAGGLE_DATA_DIR.exists() else Path.cwd() / "data"

df = pd.read_csv(DATA_DIR / "train.csv")

MODEL = "bert-base-uncased"
tok = AutoTokenizer.from_pretrained(MODEL)
label_map = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

# ---- Q1: label encoding ----
df["label"] = df["answer"].map(label_map)
print("Q1 label at idx 150:", df.loc[150, "label"])

# ---- Q2: prompt-option formatting ----
row0 = df.iloc[0]
opt_b_input = str(row0["prompt"]) + " [SEP] " + str(row0["B"])
print("Q2 formatted length:", len(opt_b_input))

# ---- helper: tokenize one row as 5 choices ----
def tokenize_row(row, max_length=128):
    options = ["A", "B", "C", "D", "E"]
    prompts = [str(row["prompt"])] * 5
    choices = [str(row[o]) for o in options]
    enc = tok(prompts, choices, padding="max_length", truncation=True,
              max_length=max_length, return_tensors="pt")
    input_ids = enc["input_ids"].unsqueeze(0).to(device)        # [1, 5, L]
    attention_mask = enc["attention_mask"].unsqueeze(0).to(device)
    return input_ids, attention_mask

# ---- Q3 ----
input_ids, attention_mask = tokenize_row(row0, 128)
print("Q3 shape:", input_ids.shape)  # [1, 5, 128]

# ---- Q4: batch of first 16 rows ----
batch_ids = []
for i in range(16):
    ids, _ = tokenize_row(df.iloc[i], 128)
    batch_ids.append(ids.squeeze(0))
batch_ids = torch.stack(batch_ids)  # [16, 5, 128]
print("Q4 shape:", batch_ids.shape, "total positions:", batch_ids.numel())

# ---- Q5 & Q6: model outputs ----
model = AutoModelForMultipleChoice.from_pretrained(MODEL).to(device)
model.eval()
with torch.no_grad():
    out = model(input_ids=input_ids, attention_mask=attention_mask)
print("Q5 logits shape:", out.logits.shape)

label_tensor = torch.tensor([row0["label"]]).to(device)
with torch.no_grad():
    out_loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=label_tensor)
print("Q6 loss ndim:", out_loss.loss.dim(), "loss value:", out_loss.loss.item())

# ---- Q7: LoRA trainable params ----
lora_cfg = LoraConfig(
    r=8, lora_alpha=16, target_modules=["query", "value"],
    lora_dropout=0.1, bias="none", task_type=TaskType.SEQ_CLS,
)
lora_model = get_peft_model(AutoModelForMultipleChoice.from_pretrained(MODEL), lora_cfg).to(device)
trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
print("Q7 trainable params:", trainable)

# ---- Q8: HF Dataset for first 100 rows ----
def build_example(i):
    row = df.iloc[i]
    ids, mask = tokenize_row(row, 128)
    return {
        "input_ids": ids.squeeze(0).tolist(),
        "attention_mask": mask.squeeze(0).tolist(),
        "labels": int(row["label"]),
    }

examples = [build_example(i) for i in range(100)]
hf_ds = Dataset.from_list(examples)
item0 = hf_ds[0]
print("Q8 input_ids shape:", (len(item0["input_ids"]), len(item0["input_ids"][0])))

# ---- Q9: tiny LoRA fine-tune (spec says max_length=64) ----
def build_example_64(i):
    row = df.iloc[i]
    ids, mask = tokenize_row(row, max_length=64)
    return {
        "input_ids": ids.squeeze(0).tolist(),
        "attention_mask": mask.squeeze(0).tolist(),
        "labels": int(row["label"]),
    }

train_examples = [build_example_64(i) for i in range(32)]
train_ds = Dataset.from_list(train_examples)
train_ds.set_format(type="torch")

lora_model2 = get_peft_model(AutoModelForMultipleChoice.from_pretrained(MODEL), lora_cfg)

OUTPUT_DIR = Path("/kaggle/working/mcq_out") if Path("/kaggle/working").exists() else Path.cwd() / "mcq_out"

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    seed=42,
    report_to=[],
)

trainer = Trainer(model=lora_model2, args=args, train_dataset=train_ds)
train_out = trainer.train()
print("Q9 global_step:", trainer.state.global_step)

lora_model2 = trainer.model.to(device)

# ---- Q10: probability of option E after fine-tuning ----
# use max_length=64 to match the fine-tuning input shape
input_ids_64, attention_mask_64 = tokenize_row(row0, max_length=64)

lora_model2.eval()
with torch.no_grad():
    out2 = lora_model2(input_ids=input_ids_64, attention_mask=attention_mask_64)
probs = torch.softmax(out2.logits, dim=-1)
print("Q10 P(E):", round(probs[0, 4].item(), 4))

Using device: cuda


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Q7 trainable params: 295681
Q8 input_ids shape: (5, 128)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`use_return_dict` is deprecated! Use `return_

Step,Training Loss
1,3.196341
2,3.162339
3,3.390414
4,3.196771


Q9 global_step: 4
Q10 P(E): 0.2117
